# Pengumpulan Tugas UTS
**Mata Kuliah:** Teknik Kompilasi
**Nama:** Rizky Rangga Baskara Lubis
**NIM:** 231011401836
**Kelas:** 06TPLM002



### 1. Definisi Node AST
Bagian ini mendefinisikan struktur data pohon untuk representasi kode.

In [ ]:
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass

### 2. Implementasi MiniCompiler
Lengkapi bagian bertanda `TUGAS` di bawah ini.

In [ ]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1:
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        """Menangani unit terkecil: angka, variabel, atau ekspresi dalam kurung."""
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    # TUGAS 2:
    def power(self):
        """Menangani operator pangkat '^' dengan prioritas di atas * dan /."""
        node = self.factor()
        while self._current == '^':
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.factor())
        return node

    def term(self):
        # TUGAS 3:
        node = self.power()  
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power())  
        return node

    def expr(self):
        """Menangani penjumlahan dan pengurangan (prioritas terendah)."""
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        """Menghasilkan Three Address Code (TAC) dari AST secara rekursif."""
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name

        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)

        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name

### 3. Uji Coba
Gunakan sel ini untuk menguji implementasi Anda.

In [ ]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
except Exception as e:
    print(f"Error: {e}")

**Output yang didapatkan:**
```
Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2
```

In [ ]:
# Uji coba tambahan

# 1: Ekspresi dengan beberapa operator
print("=== Test 1: a ^ 2 * b + c ===")
compiler2 = MiniCompiler("a ^ 2 * b + c", {'a': 3, 'b': 2, 'c': 1})
compiler2.generate_tac(compiler2.expr())

print()

# 2: Ekspresi dengan tanda kurung
print("=== Test 2: (a + b) ^ 2 ===")
compiler3 = MiniCompiler("(a + b) ^ 2", {'a': 3, 'b': 2})
compiler3.generate_tac(compiler3.expr())

print()

# 3: Semantik error - variabel tidak terdefinisi
print("=== Test 3: Semantik Error ===")
try:
    compiler4 = MiniCompiler("z + 1", {'a': 5})
    compiler4.expr()
except ParserError as e:
    print(f"Tertangkap: {e}")

---
### 4. Pertanyaan Refleksi

**1. Mengapa fungsi `power()` harus dipanggil di dalam `term()`, bukan sebaliknya? Jelaskan kaitannya dengan *Operator Precedence*.**

Dalam Recursive Descent Parser, hierarki pemanggilan fungsi mencerminkan hierarki prioritas operator. Fungsi yang dipanggil **lebih dalam** (lebih dekat ke `factor()`) berarti memiliki prioritas **lebih tinggi**, karena ia diselesaikan terlebih dahulu sebelum hasilnya dikembalikan ke fungsi pemanggil di atas.

Hierarki pemanggilan yang dibangun adalah:
```
expr()       → menangani +, -    (prioritas terendah)
  └── term() → menangani *, /    (prioritas menengah)
        └── power() → menangani ^ (prioritas tinggi)
              └── factor() → angka, variabel, (ekspresi) (prioritas tertinggi)
```

Ketika `term()` memanggil `self.power()`, itu berarti: *"sebelum aku bisa melakukan perkalian/pembagian, selesaikan dulu semua operasi pangkat yang ada di operandku."* Jika `term()` tidak memanggil `power()` dan langsung ke `factor()`, maka ekspresi `a ^ 2 * b` akan diparsing sebagai `a ^ (2 * b)` (salah), bukan `(a ^ 2) * b` (benar).

Sebaliknya, jika `power()` memanggil `term()`, maka `^` justru memiliki prioritas **lebih rendah** dari `*` dan `/`, yang bertentangan dengan aturan matematika.

---

**2. Apa yang terjadi pada fase Analisis Semantik jika variabel `z` digunakan dalam kode sumber tetapi tidak ada di `symbol_table`?**

Pada fase **Analisis Semantik**, kompiler tidak hanya memeriksa apakah sintaks benar, tetapi juga apakah *makna* dari kode tersebut valid. Salah satu pemeriksaan semantik yang paling mendasar adalah **pengecekan deklarasi variabel** (*undeclared variable check*).

Dalam implementasi ini, `symbol_table` berperan sebagai *scope* atau lingkungan tempat variabel yang valid didaftarkan. Di dalam fungsi `factor()`, terdapat pengecekan:
```python
if token not in self._env:
    raise ParserError(f"Semantic Error: Undefined variable '{token}'")
```
Ketika parser menemukan identifier `z` dan mengecek `self._env` (symbol table), `z` tidak ditemukan. Akibatnya, proses kompilasi **dihentikan** dan dilemparkan exception `ParserError` dengan pesan `Semantic Error: Undefined variable 'z'`. Ini mencegah program menghasilkan kode TAC yang tidak valid atau menyebabkan error saat runtime.

---

**3. Jelaskan mengapa dalam TAC, instruksi untuk `a ^ 2` harus muncul sebelum instruksi untuk `+`.**

Hal ini berkaitan langsung dengan sifat **post-order traversal** pada AST dan prinsip **dependency** dalam TAC.

TAC dihasilkan oleh fungsi `generate_tac()` yang berjalan secara rekursif. Untuk sebuah `BinOp`, fungsi ini:
1. Pertama memanggil `generate_tac(node.left)` — menghasilkan semua instruksi untuk sisi kiri.
2. Kemudian memanggil `generate_tac(node.right)` — menghasilkan semua instruksi untuk sisi kanan.
3. **Baru kemudian** membuat instruksi untuk operator itu sendiri menggunakan hasil dari langkah 1 dan 2.

Untuk ekspresi `a ^ 2 + b * c`, AST-nya adalah:
```
        +
       / \
      ^   *
     / \ / \
    a  2 b  c
```
Instruksi `t1 = a ^ 2` **harus** dibuat terlebih dahulu karena nilainya (`t1`) adalah **operand** yang dibutuhkan oleh instruksi `t3 = t1 + t2`. Dalam TAC, setiap variabel sementara (temporary) harus sudah didefinisikan sebelum digunakan. Urutan ini menjamin bahwa kode yang dihasilkan dapat dieksekusi secara sekuensial oleh mesin tanpa adanya referensi ke nilai yang belum dihitung.